In [1]:
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import time

/Users/layvvs/Desktop/HSE/Studying/year-project/hse-ai-year-project-2025/music-recommendation/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def parse_list_string(s):
    if not isinstance(s, str):
        return []
    s = s.strip()
    if len(s) < 2:
        return []
    parts = s[1:-1].replace("'", '').split(',')
    return [p.strip() for p in parts if p.strip()]

text_embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [3]:
data = pd.read_csv('../bmyakin-eda/output.csv').reset_index(drop=True)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           2001 non-null   int64  
 1   name                         2001 non-null   object 
 2   duration                     2001 non-null   int64  
 3   artist_id                    2001 non-null   int64  
 4   artist_name                  2001 non-null   object 
 5   album_id                     1997 non-null   float64
 6   album_name                   1997 non-null   object 
 7   releasedate                  2001 non-null   object 
 8   waveform                     2000 non-null   object 
 9   musicinfo.vocalinstrumental  1991 non-null   object 
 10  musicinfo.lang               832 non-null    object 
 11  musicinfo.gender             1723 non-null   object 
 12  musicinfo.acousticelectric   1052 non-null   object 
 13  musicinfo.speed   

In [4]:
data = data.drop(['waveform', 'album_id', 'id', 'album_id', 'artist_id', 'musicinfo.acousticelectric', 'musicinfo.lang'], axis=1)
data = data.dropna(subset=['album_name', 'musicinfo.vocalinstrumental', 'musicinfo.speed'])
data = data.fillna(value={'musicinfo.gender': 'neutral'})
data = data.reset_index(drop=True)

In [5]:
new_col = []

for idx in range(len(data)):
    genres = parse_list_string(data.loc[idx]['musicinfo.tags.genres'])
    instruments = parse_list_string(data.loc[idx]['musicinfo.tags.instruments'])
    tags = parse_list_string(data.loc[idx]['musicinfo.tags.vartags'])

    genres = genres if genres else ['unknown']
    instruments = instruments if instruments else ['unknown']
    tags = tags if tags else ['unknown']

    genres_emb = np.mean(text_embedder.encode(genres), axis=0)
    instruments_emb = np.mean(text_embedder.encode(instruments), axis=0)
    tags_emb = np.mean(text_embedder.encode(tags), axis=0)

    ready_emb = np.concat([genres_emb, instruments_emb, tags_emb])

    new_col.append(ready_emb)

In [6]:
data['embeddings'] = new_col
data['embeddings'] = data['embeddings'].apply(lambda x: x.tolist())

time_id = int(time.time() * 1000)
data.to_csv(f'database-{time_id}.csv', index=False)

In [7]:
CSV_PATH = f'database-{time_id}.csv'
DATABASE_NAME = f'vector_database_{time_id}'
COLLECTION_NAME = 'music'

csv_database = pd.read_csv(CSV_PATH)

csv_database['embeddings'] = csv_database['embeddings'].apply(lambda x: np.array(list(map(float, x[1:-1].split(', ')))))

client = chromadb.PersistentClient(path=DATABASE_NAME)

collection = client.get_or_create_collection(name=COLLECTION_NAME)

ids = csv_database.index.astype(str).tolist()
documents = [''] * len(csv_database)
embeddings = csv_database['embeddings'].tolist()
metadatas = csv_database[[
    'name',
    'duration',
    'artist_name',
    'album_name',
    'releasedate',
    'musicinfo.vocalinstrumental',
    'musicinfo.gender',
    'musicinfo.speed',
    'musicinfo.tags.genres',
    'musicinfo.tags.instruments',
    'musicinfo.tags.vartags',
    'stats.rate_downloads_total',
    'stats.rate_listened_total',
    'stats.playlisted',
    'stats.favorited',
    'stats.likes',
    'stats.dislikes',
    'stats.avgnote'
]].to_dict(orient='records')

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

In [8]:
results = collection.query(
    query_embeddings=[csv_database.loc[0]['embeddings']],
    n_results=6
)

dist = list(map(lambda x: 1 - x, results['distances'][0]))

for d, r in zip(dist, results['metadatas'][0]):
    print(d, r['name'], type(r['musicinfo.tags.genres']), r['musicinfo.tags.instruments'], r['musicinfo.tags.vartags'])


1.0 Love Too Serious <class 'str'> ['synthesizer', 'drums'] ['energetic', 'acoustic', 'vocal', 'voice']
1.0 Everybody On Your Block <class 'str'> ['synthesizer', 'drums'] ['energetic', 'acoustic', 'vocal', 'voice']
1.0 The Devil You Know <class 'str'> ['synthesizer', 'drums'] ['energetic', 'acoustic', 'vocal', 'voice']
0.9235251694917679 Disaster <class 'str'> ['synthesizer', 'drums'] ['acoustic', 'vocal', 'fast', 'voice']
0.9214037880301476 Jealousy <class 'str'> ['synthesizer', 'drums'] ['energetic', 'acoustic', 'vocal', 'voice']
0.8591044396162033 Stitches ft. Shane MauX <class 'str'> ['synthesizer', 'drums'] ['energetic', 'acoustic', 'vocal', 'voice']
